### Expand the annotations so that each waggle run is in a separate line

The annotation file generated from the annotator tool is consolidated so that multiple waggle runs appear in a single row in csv that forms the waggle dance. But for training and evaluation this is simplied.

In [57]:
import pandas as pd
import ast
import math
import os
from pathlib import Path

# Folder containing your CSV files
# folder = Path(os.getcwd())   # convert string → Path object

folder = Path("C:/Users/prajn/Documents/Sem 4/Nieh_New_vids/trial_other")

for csv_path in folder.glob("*.csv"):
    print(f"Processing: {csv_path.name}")

    df = pd.read_csv(csv_path)

    required_cols = [
        "thorax_positions", "thorax_frames",
        "waggle_start_positions", "waggle_start_frames",
        "waggle_directions", "video_name"
    ]

    # Skip files without the correct annotation format
    if not all(col in df.columns for col in required_cols):
        print(f"Skipped (missing columns)")
        continue

    expanded_rows = []

    for _, row in df.iterrows():
        thorax_positions = ast.literal_eval(row["thorax_positions"])
        thorax_frames = ast.literal_eval(row["thorax_frames"])
        waggle_positions = ast.literal_eval(row["waggle_start_positions"])
        waggle_frames = ast.literal_eval(row["waggle_start_frames"])
        waggle_dirs = ast.literal_eval(row["waggle_directions"])

        for t_pos, t_frame, w_pos, w_frame, w_dir in zip(
                thorax_positions, thorax_frames,
                waggle_positions, waggle_frames,
                waggle_dirs):

            expanded_rows.append({
                "video_name": row["video_name"],
                "start_frame": w_frame,
                "end_frame": t_frame,
                "x1": w_pos[0],
                "y1": w_pos[1],
                "x2": t_pos[0],
                "y2": t_pos[1],
                "angle": math.atan2(w_dir[0], w_dir[1]),
                "waggle": 1,
                "origin_x": w_pos[0],
                "origin_y": w_pos[1],
                "direction_x": w_dir[0],
                "direction_y": w_dir[1]
            })

    df_expanded = pd.DataFrame(expanded_rows)

    # Output filename
    output_path = csv_path.with_name(csv_path.stem + "_expanded.csv")

    df_expanded.to_csv(output_path, index=False)
    print(f"Saved: {output_path.name}")

print("\nDone.")


Processing: T14-D1-B91-V1-E_waggle_annotations.csv
Saved: T14-D1-B91-V1-E_waggle_annotations_expanded.csv
Processing: T14-D1-B91-V2-E_waggle_annotations.csv
Saved: T14-D1-B91-V2-E_waggle_annotations_expanded.csv
Processing: T14-D1-B91-V3-E_waggle_annotations.csv
Saved: T14-D1-B91-V3-E_waggle_annotations_expanded.csv
Processing: T14-D1-B92-V1-E_waggle_annotations.csv
Saved: T14-D1-B92-V1-E_waggle_annotations_expanded.csv
Processing: T14-D1-B92-V2-E_waggle_annotations.csv
Saved: T14-D1-B92-V2-E_waggle_annotations_expanded.csv
Processing: T14-D1-B92-V3-E_waggle_annotations.csv
Saved: T14-D1-B92-V3-E_waggle_annotations_expanded.csv
Processing: T14-D1-B93-V1-E_waggle_annotations.csv
Saved: T14-D1-B93-V1-E_waggle_annotations_expanded.csv
Processing: T14-D1-B94-V1-E_waggle_annotations.csv
Saved: T14-D1-B94-V1-E_waggle_annotations_expanded.csv
Processing: T14-D1-B94-V2-E_waggle_annotations.csv
Saved: T14-D1-B94-V2-E_waggle_annotations_expanded.csv
Processing: T14-D1-B94-V3-E_waggle_annotations

Combine all videos annotations to a single csv

In [59]:
import glob
import os

pattern = "C:/Users/prajn/Documents/Sem 4/Nieh_New_vids/trial_other/*waggle_annotations_expanded.csv"  

# Get list of matching files
csv_files = glob.glob(pattern)

print(f"Found {len(csv_files)} files")

# Read and concatenate
df_list = [pd.read_csv(f) for f in csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV
output_path = "C:/Users/prajn/Documents/Sem 4/Nieh_New_vids/trial_other/nieh_new_other_data_pos_full.csv"
combined_df.to_csv(output_path, index=False)

print(f"Saved combined CSV to {output_path}")

Found 30 files
Saved combined CSV to C:/Users/prajn/Documents/Sem 4/Nieh_New_vids/trial_other/nieh_new_other_data_pos_full.csv


Add the waggle run id to each waggle run in the annotation file

In [73]:
df = pd.read_csv("C:/Users/prajn/Documents/Sem 4/Thesis/slowfast_roi/data/annotations/multires_nieh_new_other_data_pos_full.csv")

In [74]:
df["waggle_run_id"] = df.groupby("video_name").cumcount() + 1

df.to_csv("C:/Users/prajn/Documents/Sem 4/Thesis/slowfast_roi/data/annotations/multires_nieh_new_other_data_pos_full_ids.csv", index=False)

In [75]:
df

,video_name,start_frame,end_frame,x1,y1,x2,y2,angle,waggle,origin_x,origin_y,direction_x,direction_y,waggle_run_id
0,T3-D1-B14-V2-C_480_270.mp4,4252,4278,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1
1,T3-D1-B14-V2-C_480_270.mp4,4349,4392,207.0,135.5,198.0,172.0,-1.321544,1,414,271,-0.969097,0.246679,2
2,T3-D1-B14-V2-C_480_270.mp4,4455,4484,193.5,148.0,159.5,157.0,-1.146602,1,387,296,-0.911371,0.411587,3
3,T3-D1-B14-V2-C_480_270.mp4,4552,4574,175.0,169.0,172.0,186.5,-0.622909,1,350,338,-0.583400,0.812185,4
4,T3-D1-B14-V2-C_480_270.mp4,4645,4669,174.5,132.5,146.5,126.5,-1.559303,1,349,265,-0.999934,0.011493,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587,T9-D1-B62-V6-C_960_540.mp4,645,665,487.0,212.0,466.0,187.0,-2.761086,1,487,212,-0.371391,-0.928477,4
588,T9-D1-B62-V6-C_960_540.mp4,777,785,494.0,209.0,493.0,203.0,2.553590,1,494,209,0.554700,-0.832050,5
589,T9-D1-B62-V6-C_960_540.mp4,917,927,448.0,211.0,451.0,193.0,2.505084,1,448,211,0.594391,-0.804176,6
590,T9-D1-B62-V6-C_960_540.mp4,1081,1107,447.0,184.0,473.0,168.0,2.214297,1,447,184,0.800000,-0.600000,7


Interpolate the points from start frame to end frame in the direction at the start

In [76]:
import pandas as pd
import numpy as np

rows = []

for _, r in df.iterrows():
    video_name = r["video_name"]
    start_f = int(r["start_frame"])
    end_f = int(r["end_frame"])
    run_id = int(r["waggle_run_id"])
    
    x1, y1 = r["x1"], r["y1"]
    x2, y2 = r["x2"], r["y2"]

    # direction unit vector already provided
    dir_x, dir_y = r["direction_x"], r["direction_y"]

    # total movement between start and end
    total_dx = x2 - x1
    total_dy = y2 - y1

    n_frames = end_f - start_f + 1
    print(video_name, x1, x2, n_frames)
    # interpolate x,y linearly
    xs = np.linspace(x1, x2, n_frames)
    ys = np.linspace(y1, y2, n_frames)

    # list all frame numbers
    frames = np.arange(start_f, end_f + 1)

    # create output rows
    for f, x, y in zip(frames, xs, ys):
        rows.append([
            video_name,
            f,
            run_id,
            x,
            y,
            dir_x,
            dir_y
        ])

# ---- Create output dataframe ----
out_df = pd.DataFrame(rows, columns=[
    "video_name", "frame", "waggle_run_id", "x", "y", "dir_x", "dir_y"
])

# ---- Save to CSV ----
out_df.to_csv("C:/Users/prajn/Documents/Sem 4/Thesis/slowfast_roi/data/annotations/waggle_interpolated_points_nieh_new_other.csv", index=False)

print("Saved waggle_interpolated_points.csv with", len(out_df), "rows.")


T3-D1-B14-V2-C_480_270.mp4 235.5 211.5 27
T3-D1-B14-V2-C_480_270.mp4 207.0 198.0 44
T3-D1-B14-V2-C_480_270.mp4 193.5 159.5 30
T3-D1-B14-V2-C_480_270.mp4 175.0 172.0 23
T3-D1-B14-V2-C_480_270.mp4 174.5 146.5 25
T3-D1-B14-V2-C_480_270.mp4 195.0 182.5 35
T3-D1-B14-V2-C_480_270.mp4 167.5 126.5 29
T3-D1-B14-V2-C_480_270.mp4 159.5 128.0 28
T3-D1-B14-V2-C_480_270.mp4 85.0 75.5 29
T3-D1-B14-V2-C_480_270.mp4 88.0 72.0 22
T3-D1-B14-V2-C_480_270.mp4 180.0 177.5 24
T14-D1-B94-V1-E_480_270.mp4 251.5 256.0 23
T14-D1-B94-V1-E_480_270.mp4 250.5 265.0 22
T14-D1-B94-V1-E_480_270.mp4 241.5 252.5 24
T14-D1-B94-V1-E_480_270.mp4 259.5 265.0 15
T14-D1-B91-V3-E_480_270.mp4 231.0 222.5 26
T14-D1-B91-V3-E_480_270.mp4 225.5 225.0 14
T14-D1-B91-V3-E_480_270.mp4 223.0 227.5 8
T14-D1-B91-V3-E_480_270.mp4 227.0 231.0 29
T9-D1-B63-V1-C_480_270.mp4 187.5 185.5 25
T9-D1-B63-V1-C_480_270.mp4 202.5 218.0 23
T9-D1-B63-V1-C_480_270.mp4 186.5 193.5 7
T9-D1-B63-V1-C_480_270.mp4 196.0 209.0 26
T9-D1-B63-V1-C_480_270.mp4 193.5

Saved waggle_interpolated_points.csv with 14718 rows.


Create sliding windows with 16 frames clips

In [77]:
import pandas as pd
from math import ceil

def expand_waggle_windows_with_offsets_fixed16(df, stride=4, clamp_start_at_zero=True):
    """
    Returns rows with 16-frame windows that ALL overlap the original waggle.
    - stride: step size for sliding / i / j loops (default 4)
    - clamp_start_at_zero: if True, negative window starts are clamped to 0
    """
    df = df.copy()
    rows = []

    for _, row in df.iterrows():
        waggle_start = int(row['start_frame'])
        waggle_end   = int(row['end_frame'])
        dif = waggle_end - waggle_start
        if dif <= 0:
            # skip invalid intervals
            continue

        mid = dif // 2
        mid_frame = waggle_start + mid

        starts = set()  # keep unique window starts

        # ---- If waggle is short, create a centered 16-frame window ----
        if dif < 16:
            # center waggle inside a 16-frame window (integer math)
            pad_left = (16 - dif) // 2
            s = waggle_start - pad_left
            if clamp_start_at_zero:
                s = max(0, s)
            starts.add(s)

        # ---- If waggle >= 16 create sliding windows inside waggle ----
        if dif >= 16:
            # windows starting at waggle_start .. waggle_end-16 (inclusive) stepping by stride
            last_inside_start = waggle_end - 16
            s = waggle_start
            while s <= last_inside_start:
                starts.add(s if not clamp_start_at_zero else max(0, s))
                s += stride

        # ---- Backwards from the middle (your original i loop intent) ----
        i = mid_frame - dif
        while i < waggle_start:
            s = i
            if clamp_start_at_zero:
                s = max(0, s)
            # check overlap: window [s, s+16) must intersect waggle [waggle_start, waggle_end)
            if s + 16 > waggle_start and s < waggle_end:
                starts.add(s)
            i += stride

        # ---- Forwards from roughly the middle/end (your j loop intent) ----
        j = waggle_end - mid
        while j < waggle_end:
            s = j
            if clamp_start_at_zero:
                s = max(0, s)
            if s + 16 > waggle_start and s < waggle_end:
                starts.add(s)
            j += stride

        # Build rows from the unique starts (sorted for determinism)
        for s in sorted(starts):
            e = s + 16
            # clipped absolute overlap
            overlap_start = max(s, waggle_start)
            overlap_end   = min(e, waggle_end)

            # only keep windows that actually overlap (should always hold due to checks above)
            if overlap_start >= overlap_end:
                continue

            new_row = row.copy()
            new_row['start_frame'] = int(s)
            new_row['end_frame']   = int(e)
            new_row['waggle_start'] = int(waggle_start)
            new_row['waggle_end']   = int(waggle_end)
            # absolute clipped frame numbers (as you requested)
            new_row['waggle_start_in_window'] = int(overlap_start)
            new_row['waggle_end_in_window']   = int(overlap_end)

            rows.append(new_row)

    expanded_df = pd.DataFrame(rows).reset_index(drop=True)
    return expanded_df


In [78]:
new_df = expand_waggle_windows_with_offsets_fixed16(df)

In [79]:
new_df

,video_name,start_frame,end_frame,x1,y1,x2,y2,angle,waggle,origin_x,origin_y,direction_x,direction_y,waggle_run_id,waggle_start,waggle_end,waggle_start_in_window,waggle_end_in_window
0,T3-D1-B14-V2-C_480_270.mp4,4239,4255,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1,4252,4278,4252,4255
1,T3-D1-B14-V2-C_480_270.mp4,4243,4259,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1,4252,4278,4252,4259
2,T3-D1-B14-V2-C_480_270.mp4,4247,4263,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1,4252,4278,4252,4263
3,T3-D1-B14-V2-C_480_270.mp4,4251,4267,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1,4252,4278,4252,4267
4,T3-D1-B14-V2-C_480_270.mp4,4252,4268,235.5,158.5,211.5,137.0,-2.016902,1,471,317,-0.902134,-0.431455,1,4252,4278,4252,4268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5409,T9-D1-B62-V6-C_960_540.mp4,1229,1245,476.0,199.0,485.0,172.0,2.804918,1,476,199,0.330350,-0.943858,8,1225,1249,1229,1245
5410,T9-D1-B62-V6-C_960_540.mp4,1233,1249,476.0,199.0,485.0,172.0,2.804918,1,476,199,0.330350,-0.943858,8,1225,1249,1233,1249
5411,T9-D1-B62-V6-C_960_540.mp4,1237,1253,476.0,199.0,485.0,172.0,2.804918,1,476,199,0.330350,-0.943858,8,1225,1249,1237,1249
5412,T9-D1-B62-V6-C_960_540.mp4,1241,1257,476.0,199.0,485.0,172.0,2.804918,1,476,199,0.330350,-0.943858,8,1225,1249,1241,1249


In [80]:
def update_xy_with_offsets(expanded_df, offsets_df):
    # Create a lookup dict for fast access: (video_name, frame) -> (x, y)
    lookup = {
        (row.video_name, row.frame, row.waggle_run_id): (row.x, row.y)
        for row in offsets_df.itertuples(index=False)
    }

    # Update expanded_df in place
    for idx, row in expanded_df.iterrows():
        key = (row["video_name"], row["waggle_start_in_window"], row['waggle_run_id'])
        if key in lookup:
            expanded_df.at[idx, "x1"], expanded_df.at[idx, "y1"] = lookup[key]

    return expanded_df


In [81]:
new_df = update_xy_with_offsets(new_df, out_df)

In [82]:
new_df.to_csv("C:/Users/prajn/Documents/Sem 4/Thesis/slowfast_roi/data/annotations/extended_window_nieh_new_other_full.csv")

### Use the script Sample_Videos_and_Annotations.py to generate videos and annotations in different resolutions

location: /wdd_3/scripts/Sample_Videos_and_Annotations.py

### Adjust annotations for fps sampling

In [103]:
import pandas as pd
from pathlib import Path

def adjust_annotations(
    csv_path,
    original_fps=15,
    new_fps=30,
    window_size=16,
    stride=16,
    out_path=None
):
    df = pd.read_csv(csv_path)
    df["video_id"] = df["video_name"].str.extract(r"^(\d{3})")

    # Assign per-video target fps
    def choose_fps(prefix):
        if pd.isna(prefix):
            return None   # no numbers → no sampling
        vid = int(prefix)
        if 1 <= vid <= 38:
            return 60
        elif 39 <= vid <= 78:
            return 30
        return None 

    df["target_fps"] = df["video_id"].apply(choose_fps)
    out_rows = []

    for _, row in df.iterrows():
        
        target_fps = row["target_fps"]

        # If no sampling rule → skip row entirely
        if pd.isna(target_fps):
            continue

        ratio = float(target_fps) / float(original_fps)
        # --- Scale frame indices ---
        s0 = int(round(row["start_frame"] * ratio))
        e0 = int(round(row["end_frame"]   * ratio))
        ws = int(round(row["waggle_start"] * ratio))
        we = int(round(row["waggle_end"]   * ratio))

        if e0 < s0:
            s0, e0 = e0, s0

        span = e0 - s0 + 1

        def make_row(window_start, window_end):
            """Create row and apply waggle-length >=4 condition."""
            
            new_row = row.copy()
            
            # Clip waggle into the window bounds (still absolute/global frame numbers)
            waggle_start_in_window = max(window_start, min(window_end, ws))
            waggle_end_in_window   = max(window_start, min(window_end, we))

            # --- NEW RULE: Keep only if waggle occupies >= 4 frames in this window ---
            waggle_length = waggle_end_in_window - waggle_start_in_window + 1
            if waggle_length < 4:
                return None

            new_row = row.copy()
            new_row["start_frame"] = window_start
            new_row["end_frame"]   = window_end

            new_row["waggle_start"] = ws
            new_row["waggle_end"]   = we

            new_row["waggle_start_in_window"] = waggle_start_in_window
            new_row["waggle_end_in_window"]   = waggle_end_in_window
            
            orig_name = row["video_name"]
            fps = int(row["target_fps"])
            clean_name = orig_name.replace(".mp4", f"_{fps}fps.mp4")
            new_row["video_name"] = clean_name

            return new_row

        # --- Case 1: short span (< window size) ---
        if span < window_size:
            w_start = s0
            w_end = s0 + window_size
            new_row = make_row(w_start, w_end)
            if new_row is not None:
                out_rows.append(new_row)

        else:
            # --- Case 2: sliding windows ---
            last_start = e0 - window_size + 1
            starts = list(range(s0, last_start + 1, stride))

            if not starts or starts[-1] != last_start:
                starts.append(last_start)

            for st in starts:
                en = st + window_size 
                new_row = make_row(st, en)
                if new_row is not None:
                    out_rows.append(new_row)

    out_df = pd.DataFrame(out_rows)

    if out_path is None:
        p = Path(csv_path)
        out_path = p.with_name(p.stem + f"_resampled_{new_fps}fps.csv")

    out_df.to_csv(out_path, index=False)
    print(f"Saved {len(out_df)} rows to {out_path}")
    return out_df


In [104]:
adjusted = adjust_annotations(
    "C:/Users/prajn/Documents/Sem 4/Thesis/slowfast_roi/data/annotations/multires_extended_windows_data_pos_full_ids.csv",
    original_fps=15,
    new_fps=60,
    window_size=16,
    stride=16
)

Saved 20313 rows to C:\Users\prajn\Documents\Sem 4\Thesis\slowfast_roi\data\annotations\multires_extended_windows_data_pos_full_ids_resampled_60fps.csv


### Prepare expanded annotation file for evaluation using old wdd - extra fields start and end timestamps

In [3]:
import pandas as pd
import ast
import math
import os
from pathlib import Path
from datetime import datetime, timedelta


def expand_annotation_files_with_ts(csv_path, output_path, name, fps=60):

    df = pd.read_csv(csv_path)
    START_TIME = datetime.fromisoformat("2024-08-30 10:00:00+00:00")

    required_cols = [
        "thorax_positions", "thorax_frames",
        "waggle_start_positions", "waggle_start_frames",
        "waggle_directions", "video_name"
    ]

    expanded_rows = []

    for _, row in df.iterrows():
        thorax_positions = ast.literal_eval(row["thorax_positions"])
        thorax_frames = ast.literal_eval(row["thorax_frames"])
        waggle_positions = ast.literal_eval(row["waggle_start_positions"])
        waggle_frames = ast.literal_eval(row["waggle_start_frames"])
        waggle_dirs = ast.literal_eval(row["waggle_directions"])

        for t_pos, t_frame, w_pos, w_frame, w_dir in zip(
                thorax_positions, thorax_frames,
                waggle_positions, waggle_frames,
                waggle_dirs):
            
            start_ts = START_TIME + timedelta(seconds=w_frame / fps)
            end_ts = START_TIME + timedelta(seconds=t_frame / fps)

            expanded_rows.append({
                "video_name": row["video_name"],
                "start_frame": w_frame,
                "end_frame": t_frame,
                "start_ts": start_ts,
                "end_ts": end_ts,
                "x1": w_pos[0],
                "y1": w_pos[1],
                "x2": t_pos[0],
                "y2": t_pos[1],
                "angle": math.atan2(w_dir[0], w_dir[1]),
                "waggle": 1,
                "origin_x": w_pos[0],
                "origin_y": w_pos[1],
                "end_x": t_pos[0],
                "end_y": t_pos[1],
                "direction_x": w_dir[0],
                "direction_y": w_dir[1]
            })

    df_expanded = pd.DataFrame(expanded_rows)
    output_path =  os.path.join(output_path,name+"_expanded_ts.csv")
    df_expanded.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")


In [9]:
expand_annotation_files_with_ts("C:/Users/prajn/Documents/Sem 4/waggle_dance_detector-clean/data/eval_videos/T4-D3-B25-V1-E_waggle_annotations.csv", 
                                "C:/Users/prajn/Documents/Sem 4/waggle_dance_detector-clean/data/eval_annotations", "T4-D3-B25-V1-E",60)

Saved: C:/Users/prajn/Documents/Sem 4/waggle_dance_detector-clean/data/eval_annotations\T4-D3-B25-V1-E_expanded_ts.csv


In [13]:
df = pd.read_csv("C:/Users/prajn/Documents/Sem 4/waggle_dance_detector-clean/data/eval_annotations/cam-0_20250904T140739.494695.291Z--20250904T140839.480715.291Z_1024_736.csv")
START_TIME = datetime.fromisoformat("2024-08-30 10:00:00+00:00")
fps = 60
df["start_ts"] = df["start_frame"].apply(lambda w: START_TIME + timedelta(seconds=w / fps))
df["end_ts"]   = df["end_frame"].apply(lambda t: START_TIME + timedelta(seconds=t / fps))
df.to_csv("C:/Users/prajn/Documents/Sem 4/waggle_dance_detector-clean/data/eval_annotations/cam-0_20250904T140739.494695.291Z--20250904T140839.480715.291Z_1024_736_ts.csv")